In [1]:
pip install beautifulsoup4 lxml requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 56.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [3]:
PROMO_DIR = Path("data/external/walmart/promotions")
RAW_HTML_DIR = PROMO_DIR / "raw_html"
PROCESSED_DIR = PROMO_DIR / "processed"

RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "walmart_ca_flyer_snapshot.parquet"

In [6]:
# Filter

BASE_URL = (
    "https://www.walmart.ca/en/shop/"
    "weekly-flyer-features/6000196190101"
)

CATEGORY_ID = "10019"   # Grocery

In [7]:
# Session

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "Chrome/152.0 Safari/537.36"
    ),
    "Accept-Language": "en-CA,en;q=0.9"
})

# Fetch one page first


In [8]:
def fetch_page(page=1):

    params = {
        "catId": CATEGORY_ID,
        "page": page
    }

    response = session.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    print(
        "Page:",
        page,
        "Status:",
        response.status_code,
        "Bytes:",
        len(response.content)
    )

    response.raise_for_status()

    return response

In [9]:
response = fetch_page(1)

Page: 1 Status: 200 Bytes: 7545


# Save Raw HTML

In [10]:
snapshot_date = datetime.now().date().isoformat()

snapshot_dir = RAW_HTML_DIR / snapshot_date
snapshot_dir.mkdir(parents=True, exist_ok=True)

(snapshot_dir / "page_01.html").write_text(
    response.text,
    encoding="utf-8"
)

print("Saved raw HTML")

Saved raw HTML


# Parse products
# Money parser

In [11]:
def parse_money(value):

    if not value:
        return np.nan

    value = value.strip()

    if "¢" in value:
        number = re.search(r"(\d+(?:\.\d+)?)", value)

        if number:
            return float(number.group(1)) / 100

    number = re.search(
        r"(\d+(?:\.\d{1,2})?)",
        value.replace(",", "")
    )

    if number:
        return float(number.group(1))

    return np.nan

# Extract product ID

In [12]:
def extract_product_id(url):

    match = re.search(
        r"/(\d+)(?:\?|$)",
        url
    )

    return match.group(1) if match else None

# # Find the product card surrounding a product link

In [15]:


def find_card(anchor):

    parent = anchor

    for _ in range(8):

        parent = parent.parent

        if parent is None:
            break

        text = parent.get_text(
            " ",
            strip=True
        )

        if (
            "current price" in text.lower()
            and len(text) < 3500
        ):
            return parent

    return None

# Parse one page

In [16]:
def parse_flyer_page(html, page_number):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    product_links = soup.find_all(
        "a",
        href=lambda x:
            x and "/en/ip/" in x
    )

    products = {}

    for anchor in product_links:

        name = anchor.get_text(
            " ",
            strip=True
        )

        if (
            len(name) < 5
            or name.lower()
            in {
                "options",
                "add",
                "shop",
                "view"
            }
        ):
            continue

        href = anchor.get("href")

        if href.startswith("/"):
            product_url = (
                "https://www.walmart.ca"
                + href
            )
        else:
            product_url = href

        product_id = extract_product_id(
            product_url
        )

        if not product_id:
            continue

        card = find_card(anchor)

        if card is None:
            continue

        text = card.get_text(
            " ",
            strip=True
        )

        # current + previous price
        price_match = re.search(
            r"current price\s*"
            r"(?:Now\s*)?"
            r"(\$?\d+(?:\.\d{1,2})?|\d+¢)"
            r"(?:,\s*Was\s*"
            r"(\$?\d+(?:\.\d{1,2})?|\d+¢))?",
            text,
            flags=re.I
        )

        current_price = np.nan
        regular_price = np.nan

        if price_match:
            current_price = parse_money(
                price_match.group(1)
            )

            regular_price = parse_money(
                price_match.group(2)
            )

        # Savings
        save_match = re.search(
            r"You save\s*\$?(\d+(?:\.\d{1,2})?)",
            text,
            flags=re.I
        )

        savings = (
            float(save_match.group(1))
            if save_match
            else np.nan
        )

        promotion_type = (
            "Rollback"
            if "rollback" in text.lower()
            else None
        )

        products[product_id] = {
            "product_id": product_id,
            "product_name": name,
            "category": "Grocery",
            "current_price": current_price,
            "regular_price": regular_price,
            "savings": savings,
            "promotion_type": promotion_type,
            "product_url": product_url,
            "page_number": page_number
        }

    return list(products.values())

# Test page 1

In [17]:
page1_products = parse_flyer_page(
    response.text,
    1
)

print(
    "Products extracted:",
    len(page1_products)
)

display(
    pd.DataFrame(
        page1_products
    ).head(20)
)

Products extracted: 0


""


# Collect multiple pages
# Collector

In [18]:
all_products = {}

MAX_PAGES = 20

for page in range(1, MAX_PAGES + 1):

    try:

        response = fetch_page(page)

        # Stop rather than circumvent access controls
        if response.status_code in {
            403,
            429
        }:
            print(
                "Access limited. Stopping."
            )
            break

        html_path = (
            snapshot_dir /
            f"page_{page:02d}.html"
        )

        html_path.write_text(
            response.text,
            encoding="utf-8"
        )

        page_products = parse_flyer_page(
            response.text,
            page
        )

        print(
            "Extracted:",
            len(page_products)
        )

        if len(page_products) == 0:
            print(
                "No products found. Stopping."
            )
            break

        new_count = 0

        for product in page_products:

            pid = product["product_id"]

            if pid not in all_products:
                new_count += 1

            all_products[pid] = product

        print(
            "New:",
            new_count,
            "| Total:",
            len(all_products)
        )

        if new_count == 0:
            print(
                "No new products. "
                "Reached final page."
            )
            break

        time.sleep(1.5)

    except Exception as e:

        print(
            "Stopped on page",
            page,
            ":",
            e
        )

        break

Page: 1 Status: 200 Bytes: 741588
Extracted: 7
New: 7 | Total: 7
Page: 2 Status: 200 Bytes: 7545
Extracted: 0
No products found. Stopping.


# Build your promotion dataset
# DataFrame

In [19]:
promos = pd.DataFrame(
    all_products.values()
)

print(promos.shape)

display(promos.head())

(7, 9)


,product_id,product_name,category,current_price,regular_price,savings,promotion_type,product_url,page_number
0,10142121,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,0.50,Rollback,https://www.walmart.ca/en/ip/Campbell-s-Conden...,1
1,10316379,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,1.00,NaN,https://www.walmart.ca/en/ip/EGGO-Blueberry-Fl...,1
2,6000202703709,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,13.44,2.47,NaN,https://www.walmart.ca/en/ip/Quaker-Chewy-Choc...,1
3,6000199413869,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,5.61,NaN,https://www.walmart.ca/en/ip/Tim-Hortons-Origi...,1
4,6000199788886,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,3.99,NaN,https://www.walmart.ca/en/ip/Janes-Pub-Style-C...,1


# Add snapshot metadata

In [20]:
promos["snapshot_date"] = pd.Timestamp(
    snapshot_date
)

promos["country"] = "CA"

promos["region"] = None
promos["store_id"] = None

promos["source"] = (
    "Walmart Canada Weekly Flyer"
)

promos["captured_at"] = pd.Timestamp.now(
    tz="UTC"
)

# Calculate promotion features

In [21]:
promos["discount_amount"] = np.where(
    promos["regular_price"].notna(),
    promos["regular_price"]
    - promos["current_price"],
    promos["savings"]
)

promos["discount_pct"] = np.where(
    promos["regular_price"] > 0,

    (
        promos["discount_amount"]
        / promos["regular_price"]
    ) * 100,

    np.nan
)

In [22]:
promos["is_promoted"] = (
    (
        promos["regular_price"].notna()
        & (
            promos["current_price"]
            < promos["regular_price"]
        )
    )
    |
    promos["promotion_type"].notna()
).astype("int8")

# Promotion ID

In [23]:
def make_promo_id(row):

    text = (
        f"{row['snapshot_date']}|"
        f"{row['product_id']}"
    )

    return hashlib.sha1(
        text.encode()
    ).hexdigest()[:20]


promos["promo_id"] = promos.apply(
    make_promo_id,
    axis=1
)

QA 

In [24]:
print(
    "Products:",
    len(promos)
)

print(
    "Unique products:",
    promos["product_id"].nunique()
)

print(
    "Promoted:",
    promos["is_promoted"].sum()
)

print(
    "With regular price:",
    promos["regular_price"]
    .notna()
    .sum()
)

print(
    "Duplicate IDs:",
    promos["promo_id"]
    .duplicated()
    .sum()
)

Products: 7
Unique products: 7
Promoted: 5
With regular price: 4
Duplicate IDs: 0


In [25]:
display(
    promos[
        [
            "product_name",
            "current_price",
            "regular_price",
            "discount_amount",
            "discount_pct",
            "promotion_type"
        ]
    ]
    .sort_values(
        "discount_pct",
        ascending=False
    )
    .head(20)
)

,product_name,current_price,regular_price,discount_amount,discount_pct,promotion_type
4,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",5.98,9.97,3.99,40.020060,NaN
1,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",2.97,3.97,1.00,25.188917,NaN
3,Tim Hortons Original Blend Medium Roast Coffee...,19.86,25.47,5.61,22.025913,NaN
2,Quaker Chewy Chocolate Chips Granola Bars Vari...,10.97,13.44,2.47,18.377976,NaN
0,"Campbell's® Condensed Cream of Mushroom Soup, ...",97.00,NaN,0.50,NaN,Rollback
5,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,1.93,NaN,NaN,NaN,NaN
6,"Red Bull Energy Drink, 250 ml (4 pack)",10.97,NaN,NaN,NaN,NaN


In [26]:
promos.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    OUTPUT_PATH
)

Saved: data/external/walmart/promotions/processed/walmart_ca_flyer_snapshot.parquet
